In [ ]:
import json
from pathlib import Path

BASE = Path("medical_parameter_extraction_pipeline")
NON_ENRICHED = "structured_queries_symptoms_temporal.jsonl"
ENRICHED     = "structured_queries_symptoms_temporal_enriched.jsonl"
OUTPUT       = "structured_queries_parameters.jsonl"

EMPTY_PATIENT_CONTEXT = {
    "age": None, "age_group": None, "gender": None,
    "pregnancy_status": None, "comorbidities": []
}
EMPTY_CLINICAL = {
    "intent": None, "red_flag": None, "red_flag_reasons": []
}

# Index enriched records by original_text
enriched_index: dict[str, dict] = {}
with open(ENRICHED) as f:
    for line in f:
        line = line.strip()
        if line:
            record = json.loads(line)
            enriched_index[record["original_text"]] = record

# Merge: iterate non-enriched, substitute enriched where available
with open(NON_ENRICHED) as fin, open(OUTPUT, "w") as fout:
    for line in fin:
        line = line.strip()
        if not line:
            continue
        record = json.loads(line)
        key = record["original_text"]
        if key in enriched_index:
            merged = enriched_index[key]
        else:
            merged = record | {
                "patient_context": EMPTY_PATIENT_CONTEXT,
                "clinical_interpretation": EMPTY_CLINICAL,
            }
        fout.write(json.dumps(merged, ensure_ascii=False) + "\n")

print(f"Written {OUTPUT}")

FileNotFoundError: [Errno 2] No such file or directory: 'medical_parameter_extraction_pipeline/structured_queries_symptoms_temporal_enriched.jsonl'